# Swiss Vehicles

This notebook fetches data about vehicles in Switzerland from the [ASTRA](https://www.astra.admin.ch/astra/de/home/dokumentation/daten-informationsprodukte/fahrzeugdaten.html) website, and saves it as parquet file for further analysis.

In [2]:
import polars as pl

## Manually adapt

Select:

- `TYPE`: One of "stock" (all vehicles at status) or "new" (newly registred vehicles in year)
- `STATUS`: A year like 2025 for `TYPE == "new"` or a string like "20260101" otherwise. Check the homepage for available data stati
- `OUTFILE`: File path of the resulting parquet file
- `COMPRESSION`: Compression method for parquet file (e.g. "snappy", "gzip", "zstd")
- `COLS`: Which columns to fetch (only subset listed)

In [3]:
TYPE = "stock"  # "new" or "stock"
STATUS = "20260101"  # Integer like 2025 if TYPE == "new", string like "20260101" else
OUTFILE = f"{STATUS}_{TYPE}.parquet"
COMPRESSION = "zstd"  # "snappy", "gzip", "zstd" etc.
COLS = [
    "Fahrzeugart",
    "Karosserieform",
    "Marken_Code",
    "Marke",
    "Marke_und_Typ",
    "Typ1",
    "Typ2",
    "Typ3",
    "Typ4",
    "Farbe",
    "Sitzplätze",
    "Leergewicht",
    "Gesamtgewicht",
    "Hubraum",
    "Leistung",
    "Treibstoff",
    "Erstinverkehrsetzung_Jahr",
    "Erstinverkehrsetzung_Monat",
    "Erstinverkehrsetzung_Kanton",
    "Schildfarbe",
    "Schildart",
    "Inverkehrsetzung_Status_Code",
    "Inverkehrsetzung_Kanton",
    "PLZ",
    "Staat_Code",
]

### Derived URL (no adaption needed)

In [8]:
MAIN = "https://opendata.astra.admin.ch/ivzod/1000-Fahrzeuge_IVZ/"
SUB = {
    "stock": "1300-Fahrzeugbestaende/1320-Datensaetze_monatlich/1323-Vorjahrsdaten",
    "new": "1200-Neuzulassungen/1210-Datensaetze_monatlich/1213-Vorjahresdaten",
}
NAME = {"stock": "BEST", "new": "NEUZU"}
URL = f"{MAIN}{SUB[TYPE]}/{NAME[TYPE]}-{STATUS}.txt"
print(URL)

https://opendata.astra.admin.ch/ivzod/1000-Fahrzeuge_IVZ/1300-Fahrzeugbestaende/1320-Datensaetze_monatlich/1323-Vorjahrsdaten/BEST-20260101.txt


## Stream CSV to Parquet

In [9]:
df = (
    pl.scan_csv(
        URL,
        separator="\t",
        try_parse_dates=True,
        schema_overrides={"Leistung": pl.String},  # Swiss thousands separator...
        quote_char=None,
    )
    # .head(100)
    .select(COLS)
    .with_columns(
        pl.col("Leergewicht").cast(pl.Float64),
        pl.col("Gesamtgewicht").cast(pl.Float64),
        pl.col("Hubraum").cast(pl.Float64),
        pl.col("Leistung").str.replace("'", "").cast(pl.Float64),
    )
)
df.sink_parquet(OUTFILE, compression=COMPRESSION)

## Peak into data

In [10]:
pl.scan_parquet(f"*{TYPE}*.parquet", include_file_paths="status").head().collect()


Fahrzeugart,Karosserieform,Marken_Code,Marke,Marke_und_Typ,Typ1,Typ2,Typ3,Typ4,Farbe,Sitzplätze,Leergewicht,Gesamtgewicht,Hubraum,Leistung,Treibstoff,Erstinverkehrsetzung_Jahr,Erstinverkehrsetzung_Monat,Erstinverkehrsetzung_Kanton,Schildfarbe,Schildart,Inverkehrsetzung_Status_Code,Inverkehrsetzung_Kanton,PLZ,Staat_Code,status
str,str,i64,str,str,str,str,str,str,str,i64,f64,f64,f64,f64,str,i64,i64,str,str,str,str,str,str,str,str
"""Personenwagen""","""Limousine""",3155,"""ABARTH""","""ABARTH Zagato 750 GT Corsa""","""Zagato 750 GT Corsa""",null,null,null,"""Grau""",2,535.0,740.0,747.0,34.0,"""Benzin""",1957,9,"""SZ""","""Weiss (Motorwagen)""","""Normalschild""","""I""","""SZ""","""64..""","""CH""","""20250101_stock.parquet"""
"""Personenwagen""","""Limousine""",3155,"""ABARTH""","""ABARTH 1300-104 S""","""1300-104 S""",null,null,null,"""Weiss""",2,720.0,920.0,1280.0,49.6,"""Benzin""",1969,9,"""ZH""","""Weiss (Motorwagen)""","""Normalschild""","""I""","""BE""","""31..""","""CH""","""20250101_stock.parquet"""
"""Personenwagen""","""Limousine""",3155,"""ABARTH""","""ABARTH 1300-104 S""","""1300-104 S""",null,null,null,"""Rot""",2,720.0,920.0,1280.0,49.6,"""Benzin""",1970,8,"""ZH""","""Weiss (Motorwagen)""","""Normalschild""","""I""","""ZH""","""81..""","""CH""","""20250101_stock.parquet"""
"""Personenwagen""","""Limousine""",3155,"""ABARTH""","""Abarth Double Bubble Coupe""","""Double Bubble Coupe""",null,null,null,"""Rot""",2,540.0,750.0,747.0,31.6,"""Benzin""",1959,12,"""LU""","""Weiss (Motorwagen)""","""Normalschild""","""I""","""LU""","""62..""","""CH""","""20250101_stock.parquet"""
"""Personenwagen""","""Limousine""",3155,"""ABARTH""","""Abarth 500 Opening Edition""","""500 Opening Edition""",null,null,null,"""Weiss""",4,1110.0,1425.0,1368.0,99.0,"""Benzin""",2008,9,"""VD""","""Weiss (Motorwagen)""","""Normalschild""","""I""","""VD""","""11..""","""CH""","""20250101_stock.parquet"""


In [4]:
pl.scan_parquet(f"*{TYPE}*.parquet", include_file_paths="status").with_columns(
    pl.col("status").str.slice(0, len(STATUS))
).group_by(["Fahrzeugart", "status"]).len().sort(
    ["status", "len"], descending=True
).head(10).collect()

Fahrzeugart,status,len
str,str,u32
"""Personenwagen""","""20260101""",4837450
"""Motorrad""","""20260101""",720932
"""Lieferwagen""","""20260101""",453759
"""Sachentransportanhänger""","""20260101""",336718
"""Landwirt. Traktor""","""20260101""",145746
"""Leichter Motorwagen""","""20260101""",101072
"""Lastwagen""","""20260101""",46722
"""Arbeitsanhänger""","""20260101""",39708
"""Wohnanhänger""","""20260101""",36659
